In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, IntSlider, HBox, Layout, VBox, HTML, GridBox
from IPython.display import display

# ============================================================
# PARAMETERS
# ============================================================

np.random.seed(61)

M_max = 200
N_max = 1000

rho = 0.85
sigma_w = 1.0

burn_in = 300

# ============================================================
# GENERATE A STATIONARY AR(1) ENSEMBLE
# ============================================================

W = sigma_w * np.random.randn(M_max, N_max + burn_in)

X_full = np.zeros((M_max, N_max + burn_in))

for m in range(M_max):
    for n in range(1, N_max + burn_in):
        X_full[m, n] = rho * X_full[m, n - 1] + W[m, n]

X = X_full[:, burn_in:]

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_finite_psd(N=400, M=50):

    # --------------------------------------------------------
    # SELECT FINITE OBSERVATIONS
    # --------------------------------------------------------

    ensemble = X[:M, :N]

    realization = ensemble[0, :]

    # --------------------------------------------------------
    # FFT PARAMETERS
    # --------------------------------------------------------

    n_fft = 2048

    omega = np.linspace(-np.pi, np.pi, n_fft, endpoint=False)

    # --------------------------------------------------------
    # SINGLE-REALIZATION PERIODOGRAM
    # --------------------------------------------------------

    X_fft = np.fft.fftshift(
        np.fft.fft(realization, n=n_fft)
    )

    periodogram_single = (np.abs(X_fft) ** 2) / N

    # --------------------------------------------------------
    # ENSEMBLE-AVERAGED PERIODOGRAM
    # --------------------------------------------------------

    periodograms = np.zeros((M, n_fft))

    for m in range(M):

        Xm_fft = np.fft.fftshift(
            np.fft.fft(ensemble[m, :], n=n_fft)
        )

        periodograms[m, :] = (np.abs(Xm_fft) ** 2) / N

    periodogram_average = np.mean(
        periodograms,
        axis=0
    )

    # --------------------------------------------------------
    # THEORETICAL PSD OF AR(1)
    # --------------------------------------------------------

    theoretical_psd = (
        sigma_w ** 2
        /
        (
            1
            + rho ** 2
            - 2 * rho * np.cos(omega)
        )
    )

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, (ax1, ax2, ax3) = plt.subplots(
        3,
        1,
        figsize=(8.0, 6.8)
    )

    # ========================================================
    # GRAPH 1:
    # FINITE OBSERVATION
    # ========================================================

    ax1.plot(
        np.arange(N),
        realization,
        linewidth=1.0
    )

    ax1.set_xlim(
        0,
        N - 1
    )

    ax1.set_ylim(
        -6.0,
        6.0
    )

    ax1.set_xlabel(
        'Time index n',
        fontsize=11
    )

    ax1.set_ylabel(
        'x[n]',
        fontsize=11
    )

    ax1.set_title(
        f'Finite Observation of One AR(1) Realization, N = {N}',
        fontsize=12,
        pad=7
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # ========================================================
    # GRAPH 2:
    # SINGLE-REALIZATION PERIODOGRAM
    # ========================================================

    ax2.plot(
        omega,
        periodogram_single,
        linewidth=1.2
    )

    ax2.set_xlim(
        -np.pi,
        np.pi
    )

    ax2.set_ylim(
        0,
        60
    )

    ax2.set_xticks(
        [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi]
    )

    ax2.set_xticklabels(
        ['-π', '-π/2', '0', 'π/2', 'π']
    )

    ax2.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax2.set_ylabel(
        'Periodogram',
        fontsize=11
    )

    ax2.set_title(
        'Single-Realization Periodogram',
        fontsize=12,
        pad=7
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # ========================================================
    # GRAPH 3:
    # ENSEMBLE-AVERAGED PSD ESTIMATE
    # ========================================================

    ax3.plot(
        omega,
        periodogram_average,
        linewidth=1.7,
        label='Ensemble-averaged periodogram'
    )

    ax3.plot(
        omega,
        theoretical_psd,
        linestyle='--',
        linewidth=1.5,
        label='Theoretical PSD'
    )

    ax3.set_xlim(
        -np.pi,
        np.pi
    )

    ax3.set_ylim(
        0,
        60
    )

    ax3.set_xticks(
        [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi]
    )

    ax3.set_xticklabels(
        ['-π', '-π/2', '0', 'π/2', 'π']
    )

    ax3.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax3.set_ylabel(
        'PSD',
        fontsize=11
    )

    ax3.set_title(
        f'PSD Estimate from M = {M} Realizations',
        fontsize=12,
        pad=7
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    ax3.legend(
        fontsize=8,
        loc='upper right'
    )

    plt.subplots_adjust(
        left=0.12,
        right=0.97,
        top=0.96,
        bottom=0.09,
        hspace=0.55
    )

    plt.show()

# ============================================================
# SLIDERS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='100px'
)

N_slider = IntSlider(
    min=100,
    max=1000,
    step=50,
    value=400,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

M_slider = IntSlider(
    min=10,
    max=200,
    step=10,
    value=50,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# MAXIMUM VALUE LABELS
# ============================================================

N_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">1000</div>'
)

M_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">200</div>'
)

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_finite_psd,
    N=N_slider,
    M=M_slider
)

# ============================================================
# COMPACT THEORY / DOCUMENTATION
# ============================================================

theory_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 16px;
    line-height: 1.30;
    width: 1050px;
">

<div style="
    font-size: 21px;
    font-weight: bold;
    margin-bottom: 6px;
">
Finite Observation, Periodogram and Power Spectral Density
</div>

<div style="margin-bottom:5px;">
<b>Finite observation:</b> a random realization is restricted to a finite interval so that its Fourier transform can be calculated.
</div>

<div style="margin-bottom:5px;">
<b>Periodogram:</b> |X<sub>N</sub>(ω)|²/N is a finite-record estimate of the spectral power distribution.
</div>

<div style="margin-bottom:5px;">
<b>PSD:</b> the power spectral density describes the average signal power per unit frequency.
</div>

<div style="margin-bottom:5px;">
<b>Observation length N:</b> increasing N improves frequency resolution but does not by itself remove the random fluctuations of a single periodogram.
</div>

<div style="margin-bottom:5px;">
<b>Number of realizations M:</b> averaging periodograms over many realizations reduces random fluctuations and approaches the theoretical PSD.
</div>

<div>
<b>This notebook:</b> compares a finite random realization, its periodogram, and the ensemble-averaged PSD estimate of a WSS AR(1) process.
</div>

</div>
""")

# ============================================================
# EXTRA SPACE BELOW THEORY
# ============================================================

theory_block = VBox(
    [
        theory_html
    ],
    layout=Layout(
        margin='0px 0px 20px 0px'
    )
)

# ============================================================
# LEFT-ALIGNED LABELS
# ============================================================

N_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Samples N:</div>'
)

M_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Realizations M:</div>'
)

# ============================================================
# FIXED THREE-COLUMN GRID
# LABEL | SLIDER | MAXIMUM VALUE
# ============================================================

slider_grid = GridBox(
    children=[
        N_label, N_slider, N_max_label,
        M_label, M_slider, M_max_label
    ],
    layout=Layout(
        width='265px',
        grid_template_columns='110px 100px 40px',
        grid_template_rows='30px 30px',
        grid_gap='2px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS
# ============================================================

controls = VBox(
    [
        slider_grid
    ],
    layout=Layout(
        width='275px',
        min_width='275px',
        align_items='flex-start',
        justify_content='center',
        margin='0px 0px 0px 10px',
        overflow='hidden'
    )
)

# ============================================================
# FIGURE LEFT - CONTROLS RIGHT
# ============================================================

graph_and_controls = HBox(
    [
        widget_plot.children[-1],
        controls
    ],
    layout=Layout(
        width='1100px',
        align_items='center',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        theory_block,
        graph_and_controls
    ],
    layout=Layout(
        width='1100px',
        overflow='hidden'
    )
)

display(main_layout)